# Transformers

Build a small Transformer block from PyTorch layers. Trace token IDs through embeddings, position vectors, attention, a feed-forward step, and an output head. The goal is to inspect shapes and masks, not claim trained performance.

In [ ]:
import torch
from torch import nn

torch.manual_seed(27)
torch.set_num_threads(1)
print('CPU | small Transformer block')

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, width=8, heads=2):
        super().__init__()
        self.norm1 = nn.LayerNorm(width)
        self.attention = nn.MultiheadAttention(width, heads, batch_first=True)
        self.norm2 = nn.LayerNorm(width)
        self.feed_forward = nn.Sequential(nn.Linear(width, 16), nn.GELU(), nn.Linear(16, width))

    def forward(self, x, mask=None):
        normalized = self.norm1(x)
        attended, _ = self.attention(normalized, normalized, normalized, attn_mask=mask, need_weights=False)
        x = x + attended
        return x + self.feed_forward(self.norm2(x))

vocabulary_size, max_length, width = 12, 6, 8
embedding = nn.Embedding(vocabulary_size, width)
position = nn.Embedding(max_length, width)
block = TransformerBlock()
head = nn.Linear(width, vocabulary_size)
token_ids = torch.tensor([[1, 5, 3, 2]])
positions = torch.arange(token_ids.shape[1]).unsqueeze(0)